In [1]:
from torch.utils.tensorboard import SummaryWriter
import time
import gymnasium as gym
import torch
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributions as D
import torch.optim as optim
import imageio
import matplotlib.pyplot as plt

In [2]:

class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )

        self.mean = nn.Linear(128, action_dim)

        # global (state-independent) log std
        self.log_std = nn.Parameter(torch.zeros(action_dim))

    def forward(self, state):
        # stabilize input scale
        state = torch.clamp(state, -5.0, 5.0)

        x = self.net(state)

        mean = self.mean(x)
        log_std = self.log_std.expand_as(mean)
        log_std = torch.clamp(log_std, -20, 2)

        std = torch.exp(log_std)

        dist = D.Normal(mean, std)
        return dist


class Critic(nn.Module):
    def __init__(self, state_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, state):
        # stabilize input scale
        state = torch.clamp(state, -5.0, 5.0)
        return self.net(state)


In [3]:
class Agent():
    def __init__(self, gamma=0.95, n_steps=3,
                 entropy_coef=0.01, entropy_decay=0.995,
                 env_name="Pendulum-v1"):

        self.env = gym.make(env_name)
        state_dim = self.env.observation_space.shape[0]
        action_dim = self.env.action_space.shape[0]

        self.actor = Actor(state_dim, action_dim)
        self.critic = Critic(state_dim)

        self.action_low = torch.tensor(self.env.action_space.low, dtype=torch.float32)
        self.action_high = torch.tensor(self.env.action_space.high, dtype=torch.float32)

        self.actor_opt = optim.Adam(self.actor.parameters(), lr=1e-3)
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=3e-3)

        self.gamma = gamma
        self.n_steps = n_steps
        self.entropy_coef = entropy_coef
        self.entropy_decay = entropy_decay

        # n-step buffers
        self.states = []
        self.rewards = []
        self.log_probs = []
        self.entropies = []

        # episode stats
        self.ep_reward = 0.0
        self.ep_length = 0

        # -------- logging history --------
        self.history_reward = []
        self.history_actor_loss = []
        self.history_critic_loss = []
        self.history_entropy = []
        self.history_value = []
        # ---------------------------------
        run_name = f"A2C_Pendulum_{int(time.time())}"
        self.writer = SummaryWriter(log_dir=f"runs/{run_name}")

        self.global_step = 0
        self.global_episode = 0


    # --- training remains mostly unchanged ---
    def train(self, state, log_probs, entropy, reward, next_state, done):

        self.states.append(
            torch.tensor(state, dtype=torch.float32).unsqueeze(0).detach()
        )
        self.rewards.append(float(reward))
        self.log_probs.append(log_probs)
        self.entropies.append(entropy)

        if len(self.rewards) < self.n_steps and not done:
            return

        with torch.no_grad():
            next_state = torch.tensor(next_state, dtype=torch.float32).unsqueeze(0)
            R = 0.0 if done else self.critic(next_state).item()

        returns = []
        for r in reversed(self.rewards):
            R = r + self.gamma * R
            returns.insert(0, R)

        returns = torch.tensor(returns)
        # returns = (returns - returns.mean()) / (returns.std() + 1e-8)


        state_tensor = torch.cat(self.states, dim=0)
        values = self.critic(state_tensor).squeeze()

        advantages = returns - values
        # Normalize ONLY advantages for policy gradient stability
        if len(advantages) > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        log_probs = torch.stack(self.log_probs)
        entropies = torch.stack(self.entropies)

        # Actor loss using normalized advantages
        actor_loss = -(log_probs * advantages.detach()).mean() \
                     - self.entropy_coef * entropies.mean() 

        # Critic loss using raw returns
        critic_loss = (returns.detach() - values).pow(2).mean()

        # -------- store logs --------
        self.history_actor_loss.append(actor_loss.item())
        self.history_critic_loss.append(critic_loss.item())
        self.history_value.append(values.mean().item())

        # -------- TensorBoard logging (per update) --------
        self.writer.add_scalar("loss/actor", actor_loss.item(), self.global_step)
        self.writer.add_scalar("loss/critic", critic_loss.item(), self.global_step)
        self.writer.add_scalar("policy/entropy", entropies.mean().item(), self.global_step)
        self.writer.add_scalar("value/V_mean", values.mean().item(), self.global_step)
        self.writer.add_scalar("policy/entropy_coef", self.entropy_coef, self.global_step)

        self.global_step += 1
        # ---------------------------

        # Inside your train method
        self.actor_opt.zero_grad()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.actor.parameters(), max_norm=0.5) # Add this
        self.actor_opt.step()

        self.critic_opt.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.critic.parameters(), max_norm=0.5) # Add this
        self.critic_opt.step()

        # if done:
        #     self.entropy_coef = max(0.001, self.entropy_coef * self.entropy_decay)

        self.states.clear()
        self.rewards.clear()
        self.log_probs.clear()
        self.entropies.clear()

    # --- stochastic action for training ---
    def select_action(self, state):
        state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)

        dist = self.actor(state)
        raw_action = dist.rsample()
        action = torch.tanh(raw_action)

        action_env = self.action_low + (action + 1) * 0.5 * (
            self.action_high - self.action_low
        )

        log_prob = dist.log_prob(raw_action)
        log_prob -= torch.log(1 - torch.tanh(raw_action).pow(2) + 1e-6)
        log_prob = log_prob.sum()

        entropy = dist.entropy().sum()

        return action_env.squeeze(0).detach().numpy(), log_prob, entropy

    # --- deterministic action for evaluation (safe, no noise) ---
    def select_action_eval(self, state):
        state = torch.tensor(state, dtype=torch.float32).view(1, -1)  # <-- fixed shape

        with torch.no_grad():
            dist = self.actor(state)
            raw_action = dist.mean              # deterministic
            action = torch.tanh(raw_action)    # squash

            # scale to env range
            action_env = self.action_low + (action + 1) * 0.5 * (
                self.action_high - self.action_low
            )

        return action_env.squeeze(0).cpu().numpy()  # correct shape

    # --- evaluation, with optional RGB array rendering for GIFs ---
    def evaluate(self, num_episodes=10, render=False, save_frames=False):
        rewards = []
        all_frames = []

        for ep in range(num_episodes):
            state, _ = self.env.reset()
            done = False
            ep_reward = 0.0
            frames = []

            while not done:
                if render:
                    self.env.render()  # for human display

                action = self.select_action_eval(state)
                next_state, reward, terminated, truncated, _ = self.env.step(action)
                done = terminated or truncated

                ep_reward += reward
                state = next_state

                if save_frames:
                    frame = self.env.render(mode='rgb_array')
                    frames.append(frame)

            rewards.append(ep_reward)
            if save_frames:
                all_frames.append(frames)

            print(f"[EVAL] Episode {ep:02d} | Reward: {ep_reward:.2f}")

        avg_reward = np.mean(rewards)
        std_reward = np.std(rewards)
        print(f"\n[EVAL SUMMARY] Avg Reward: {avg_reward:.2f} ± {std_reward:.2f}")

        if save_frames:
            return avg_reward, all_frames
        return avg_reward

    # --- training loop ---
    def run(self, number_of_episodes=200):
        for ep in range(number_of_episodes):
            state, _ = self.env.reset()
            done = False

            self.ep_reward = 0.0
            self.ep_length = 0

            while not done:
                action, log_prob, entropy = self.select_action(state)
                next_state, reward, terminated, truncated, _ = self.env.step(action)
                done = terminated or truncated

                self.ep_reward += reward
                self.ep_length += 1
                self.history_entropy.append(entropy.item())

                self.train(state, log_prob, entropy, reward, next_state, done)
                state = next_state

            self.history_reward.append(self.ep_reward)
            self.writer.add_scalar("episode/reward", self.ep_reward, self.global_episode)
            self.writer.add_scalar("episode/length", self.ep_length, self.global_episode)

            self.global_episode += 1
            self.entropy_coef = max(0.0001, self.entropy_coef * self.entropy_decay)


            if ep % 10 == 0:
                print(
                    f"Episode {ep:03d} | "
                    f"Reward: {self.ep_reward:8.2f} | "
                    f"Len: {self.ep_length:4d} | "
                    f"EntropyCoef: {self.entropy_coef:.4f}"
                )

            if ep % 20 == 0:
                for name, param in self.actor.named_parameters():
                    self.writer.add_histogram(f"actor/{name}", param, ep)

                for name, param in self.critic.named_parameters():
                    self.writer.add_histogram(f"critic/{name}", param, ep)

            # if ep % 100 == 0 and ep > 0:
            #     self.plot_training()
        self.writer.close()


    # --- plotting ---
    def plot_training(self):
        fig, axs = plt.subplots(2, 1, figsize=(6, 8))

        axs[0].plot(self.history_reward)
        axs[0].set_title("Episode Reward")
        axs[1].plot(self.history_value)
        axs[1].set_title("Mean Value Estimate V(s)")

        plt.tight_layout()
        plt.show()


In [4]:
agent = Agent(gamma = 0.95, n_steps=20, entropy_coef=0.1, entropy_decay=0.995, env_name="Pendulum-v1")

In [19]:
agent.run(500)

Episode 000 | Reward:  -657.51 | Len:  200 | EntropyCoef: 0.0001
Episode 010 | Reward:  -518.16 | Len:  200 | EntropyCoef: 0.0001
Episode 020 | Reward:  -404.65 | Len:  200 | EntropyCoef: 0.0001
Episode 030 | Reward:  -540.83 | Len:  200 | EntropyCoef: 0.0001
Episode 040 | Reward:  -542.57 | Len:  200 | EntropyCoef: 0.0001
Episode 050 | Reward:  -511.66 | Len:  200 | EntropyCoef: 0.0001
Episode 060 | Reward:  -273.27 | Len:  200 | EntropyCoef: 0.0001
Episode 070 | Reward: -1076.32 | Len:  200 | EntropyCoef: 0.0001
Episode 080 | Reward:  -511.00 | Len:  200 | EntropyCoef: 0.0001
Episode 090 | Reward: -1058.42 | Len:  200 | EntropyCoef: 0.0001
Episode 100 | Reward:  -784.81 | Len:  200 | EntropyCoef: 0.0001
Episode 110 | Reward:  -395.94 | Len:  200 | EntropyCoef: 0.0001
Episode 120 | Reward:  -397.80 | Len:  200 | EntropyCoef: 0.0001
Episode 130 | Reward:  -269.43 | Len:  200 | EntropyCoef: 0.0001
Episode 140 | Reward:  -131.84 | Len:  200 | EntropyCoef: 0.0001
Episode 150 | Reward:  -3

In [26]:
agent.evaluate(10, render=False)

[EVAL] Episode 00 | Reward: -129.49
[EVAL] Episode 01 | Reward: -9.19
[EVAL] Episode 02 | Reward: -138.31
[EVAL] Episode 03 | Reward: -249.63
[EVAL] Episode 04 | Reward: -131.49
[EVAL] Episode 05 | Reward: -278.63
[EVAL] Episode 06 | Reward: -131.12
[EVAL] Episode 07 | Reward: -244.42
[EVAL] Episode 08 | Reward: -127.54
[EVAL] Episode 09 | Reward: -132.01

[EVAL SUMMARY] Avg Reward: -157.18 ± 75.35


np.float64(-157.18283291319918)

In [27]:
import os
import torch

os.makedirs("checkpoints", exist_ok=True)

torch.save(
    {
        "actor": agent.actor.state_dict(),
        "critic": agent.critic.state_dict(),
    },
    "checkpoints/pendulum_actor_critic_avg_-157.pt"
)

print("💾 Actor + Critic saved together")


💾 Actor + Critic saved together


In [5]:
checkpoint = torch.load("checkpoints/pendulum_actor_critic_avg_-157.pt")

agent.actor.load_state_dict(checkpoint["actor"])
agent.critic.load_state_dict(checkpoint["critic"])

agent.actor.eval()
agent.critic.eval()


C:\Users\91930\AppData\Local\Temp\ipykernel_4520\2419492197.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("checkpoints/pendulum_actor_critic_av

Critic(
  (net): Sequential(
    (0): Linear(in_features=3, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [7]:
agent.actor.eval()
agent.critic.eval()

rewards = []

for ep in range(10):
    state, _ = agent.env.reset()
    done = False
    ep_reward = 0.0

    while not done:
        with torch.no_grad():
            action = agent.select_action_eval(state)

        next_state, reward, terminated, truncated, _ = agent.env.step(action)
        done = terminated or truncated

        ep_reward += reward
        state = next_state

    rewards.append(ep_reward)
    print(f"[EVAL] Episode {ep:02d} | Reward: {ep_reward:.2f}")

avg = np.mean(rewards)
std = np.std(rewards)

print(f"\n[EVAL SUMMARY] Avg Reward: {avg:.2f} ± {std:.2f}")


[EVAL] Episode 00 | Reward: -127.33
[EVAL] Episode 01 | Reward: -131.59
[EVAL] Episode 02 | Reward: -275.50
[EVAL] Episode 03 | Reward: -134.29
[EVAL] Episode 04 | Reward: -243.59
[EVAL] Episode 05 | Reward: -131.43
[EVAL] Episode 06 | Reward: -120.33
[EVAL] Episode 07 | Reward: -132.41
[EVAL] Episode 08 | Reward: -129.83
[EVAL] Episode 09 | Reward: -126.70

[EVAL SUMMARY] Avg Reward: -155.30 ± 52.74


In [24]:
render_env = gym.make("Pendulum-v1", render_mode="rgb_array")

frames = []
state, _ = render_env.reset()

done = False
total_reward = 0.0

while not done:
    state_tensor = torch.as_tensor(state, dtype=torch.float32).view(1, -1)

    with torch.no_grad():
        dist = agent.actor(state_tensor)
        action_tensor = torch.tanh(dist.mean) * 2.0

    action = action_tensor.squeeze(0).cpu().numpy()

    state, reward, terminated, truncated, _ = render_env.step(action)
    done = terminated or truncated
    total_reward += reward

    frame = render_env.render()
    frames.append(frame)

render_env.close()
print("Episode reward:", total_reward)




Episode reward: -134.70352623930688


In [25]:
imageio.mimsave(
    "pendulum_eval.gif",
    frames,
    fps=30
)
print("Saved evaluation GIF. Total Reward:", total_reward)

Saved evaluation GIF. Total Reward: -134.70352623930688
